# 01 — Data Quality Checks

Systematic check of all 5 tables before building Part 1 SQL queries.
Findings get logged in `notes/data_quality_log.md`.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.db import get_connection

conn = get_connection()
pd.set_option('display.max_columns', None)

In [2]:
tables = ['player_profile', 'player_day', 'purchase', 'currency_spend', 'ad_view']

for t in tables:
    df = pd.read_sql_query(f"SELECT * FROM {t} LIMIT 5", conn)
    count = pd.read_sql_query(f"SELECT COUNT(*) as n FROM {t}", conn)['n'][0]
    print(f"--- {t} ({count:,} rows) ---")
    print(df.dtypes)
    print()

--- player_profile (80,295 rows) ---
device_id               int64
original_device_id     object
install_date              str
platform                  str
country                   str
country_tier            int64
acquisition_channel       str
first_purchase_date       str
dtype: object

--- player_day (667,435 rows) ---
device_id             int64
activity_date         int64
player_level          int64
session_count         int64
playtime_minutes    float64
app_version             str
dtype: object

--- purchase (15,426 rows) ---
purchase_id             str
device_id             int64
event_ts                str
pack_name               str
usd_amount          float64
currency_granted      int64
sale_flag             int64
dtype: object

--- currency_spend (489,892 rows) ---
device_id          int64
event_ts             str
spend_type           str
currency_type        str
currency_value     int64
spend_category       str
player_level       int64
premium_balance    int64
social_bala

## player_profile checks
Should be one row per player (unique `device_id`), sensible date ranges,
clean categorical values.

In [3]:
q = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT device_id) as distinct_devices,
    MIN(install_date) as min_install,
    MAX(install_date) as max_install,
    SUM(CASE WHEN original_device_id IS NOT NULL AND original_device_id != '' THEN 1 ELSE 0 END) as linked_devices,
    SUM(CASE WHEN first_purchase_date IS NOT NULL THEN 1 ELSE 0 END) as ever_purchased
FROM player_profile
"""
pd.read_sql_query(q, conn)

,total_rows,distinct_devices,min_install,max_install,linked_devices,ever_purchased
0,80295,80295,2025-05-27,2026-04-30,3211,11054


In [23]:
# Check install-volume trend across the 120-day analysis window
q = """
SELECT
    substr(install_date, 1, 7) AS install_month,
    COUNT(*) AS installs
FROM player_profile
WHERE install_date BETWEEN '2026-01-01' AND '2026-04-30'
GROUP BY substr(install_date, 1, 7)
ORDER BY install_month
"""
install_volume = pd.read_sql_query(q, conn)
display(install_volume)

,install_month,installs
0,2026-01,9809
1,2026-02,9471
2,2026-03,18209
3,2026-04,26806


In [13]:
q = """
SELECT
    COUNT(*) AS linked_devices,
    SUM(CASE WHEN original_device_id = device_id THEN 1 ELSE 0 END) AS self_links,
    SUM(
        CASE WHEN original_device_id IS NOT NULL
              AND original_device_id NOT IN (
                  SELECT device_id FROM player_profile
              )
        THEN 1 ELSE 0 END
    ) AS original_id_not_in_profile
FROM player_profile
"""
display(pd.read_sql_query(q, conn))

,linked_devices,self_links,original_id_not_in_profile
0,80295,1,0


In [14]:
for col in ['platform', 'country_tier', 'acquisition_channel']:
    print(f"--- {col} ---")
    print(pd.read_sql_query(f"SELECT {col}, COUNT(*) as n FROM player_profile GROUP BY {col} ORDER BY n DESC", conn))
    print()

--- platform ---
  platform      n
0  Android  57136
1      iOS  23159

--- country_tier ---
   country_tier      n
0             4  32554
1             1  18596
2             3  15081
3             2  14064

--- acquisition_channel ---
  acquisition_channel      n
0          paid_video  34036
1             organic  24554
2         paid_social  21705



## player_day checks
Grain should be one row per (device_id, activity_date). Check for dupes
and date range.

In [15]:
q = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT device_id || '_' || activity_date) as distinct_player_days,
    MIN(activity_date) as min_date,
    MAX(activity_date) as max_date
FROM player_day
"""
pd.read_sql_query(q, conn)

,total_rows,distinct_player_days,min_date,max_date
0,667435,667435,20260101,20260430


In [16]:
q = """
SELECT
    MIN(playtime_minutes) as min_playtime, MAX(playtime_minutes) as max_playtime,
    MIN(session_count) as min_sessions, MAX(session_count) as max_sessions,
    SUM(CASE WHEN playtime_minutes < 0 OR session_count < 0 THEN 1 ELSE 0 END) as negative_rows
FROM player_day
"""
pd.read_sql_query(q, conn)

,min_playtime,max_playtime,min_sessions,max_sessions,negative_rows
0,0.0,239.8,1,18,0


## purchase checks
Known issue: `purchase_id` is not unique (see notes/data_quality_log.md #1).
Check amounts, sale_flag values, and whether purchasers always have a
matching player_day row.

In [17]:
q = """
SELECT COUNT(*) as total_rows, COUNT(DISTINCT purchase_id) as distinct_ids,
       MIN(usd_amount) as min_amt, MAX(usd_amount) as max_amt,
       SUM(CASE WHEN usd_amount <= 0 THEN 1 ELSE 0 END) as non_positive_amounts
FROM purchase
"""
pd.read_sql_query(q, conn)

,total_rows,distinct_ids,min_amt,max_amt,non_positive_amounts
0,15426,15304,-99.99,99.99,228


In [19]:
q = """
SELECT COUNT(*) FROM (
  SELECT DISTINCT device_id, substr(event_ts,1,10) as pdate FROM purchase
) pu
LEFT JOIN player_day pd
  ON pu.device_id = pd.device_id
  AND pd.activity_date = CAST(replace(pu.pdate,'-','') AS INTEGER)
WHERE pd.device_id IS NULL
"""
missing = pd.read_sql_query(q, conn).iloc[0,0]
total = pd.read_sql_query("SELECT COUNT(DISTINCT device_id || substr(event_ts,1,10)) FROM purchase", conn).iloc[0,0]
print(f"{missing} of {total} purchase-days ({missing/total:.1%}) have no matching player_day row")

2611 of 15165 purchase-days (17.2%) have no matching player_day row


## currency_spend checks
Check `spend_type` / `currency_type` values, and whether balances ever go negative.

In [20]:
print(pd.read_sql_query("SELECT spend_type, COUNT(*) n FROM currency_spend GROUP BY spend_type", conn))
print(pd.read_sql_query("SELECT currency_type, COUNT(*) n FROM currency_spend GROUP BY currency_type", conn))

q = """
SELECT SUM(CASE WHEN premium_balance < 0 THEN 1 ELSE 0 END) as neg_premium,
       SUM(CASE WHEN social_balance < 0 THEN 1 ELSE 0 END) as neg_social,
       SUM(CASE WHEN grind_balance < 0 THEN 1 ELSE 0 END) as neg_grind
FROM currency_spend
"""
pd.read_sql_query(q, conn)

     spend_type       n
0          free  452299
1          paid    8933
2  subscription   28660
  currency_type       n
0         GRIND  274101
1       PREMIUM   98750
2        SOCIAL  117041


,neg_premium,neg_social,neg_grind
0,0,0,0


## ad_view checks
Check `status` values, and whether `ad_daily_count` ever exceeds `ad_daily_cap`
(would indicate a cap-enforcement bug or a logging artefact).

In [21]:
print(pd.read_sql_query("SELECT status, COUNT(*) n FROM ad_view GROUP BY status", conn))

q = """
SELECT COUNT(*) as rows_over_cap
FROM ad_view
WHERE ad_daily_count > ad_daily_cap
"""
pd.read_sql_query(q, conn)

      status       n
0  abandoned   84420
1  completed  531074
2    started   32129


,rows_over_cap
0,0


In [22]:
pd.read_sql_query("SELECT ad_daily_cap, COUNT(DISTINCT device_id) as players FROM ad_view GROUP BY ad_daily_cap ORDER BY ad_daily_cap", conn)

,ad_daily_cap,players
0,5,70588
1,8,49005


## Summary

The data-quality review identified several issues that materially affect the analysis:

- `purchase_id` is not fully unique, so it is not treated as a unique transaction identifier.
- Some purchase-days have no matching `player_day` activity record; purchase activity is therefore considered where relevant when defining active players.
- Install volume changes substantially across the analysis window, so blended metrics are interpreted with changing player composition in mind.
- Some players installed before the analysis window, limiting complete retention observation for those cohorts.
- A small number of purchases occur before the recorded install date, so negative time-to-first-purchase observations are treated as a data-quality issue rather than valid behaviour.
- Rewarded-ad data contains recurring tracking gaps and a full-day outage, which are treated as telemetry issues rather than genuine drops in engagement.